# 03 — NLP block: prompt evaluation

**Goal.** The NLP block has two distinct LLM calls:

1. **Extraction** — convert free-text growing conditions into a 7-key JSON `{N, P, K, temperature, humidity, pH, rainfall}` so the numeric model can consume them. In the app, `pH` is shown externally; before prediction, the app creates an internal `ph` compatibility key for the trained numeric model.
2. **Treatment plan generation** — use the disease label from the CV block, the extracted environment values, and the risk score from the numeric model to write a tailored treatment recommendation.

This notebook documents how we evaluated and **iterated on the prompts**. The course requires at least one prompt/model/approach comparison for the NLP block, so this notebook compares:

- Prompt A vs Prompt B for environment extraction.
- Prompt C vs Prompt D for treatment-plan generation.


In [26]:
import json
import pandas as pd

pd.set_option("display.max_colwidth", 120)


## 1. Test inputs

The extraction prompt was checked on five hand-crafted growing-condition descriptions. They cover English and German phrasing, explicit numeric values, missing values, and informal descriptions.

In [27]:
TEST_INPUTS = [
    "Tomatoes outdoors in Zurich, ~18°C, 92% humidity, light rain almost every day, soil pH 6.0, low fertiliser.",
    "Greenhouse tomato, 25 degrees, 65% rH, pH 6.5, balanced fertilisation, no rain.",
    "Potato field, warm sunny week, 27C, 80%, pH 5.9, last rainfall 130mm, N around 100 ppm.",
    "Apfelbaum im Frühling, 17 Grad, 95% Luftfeuchte, pH 6.1, viel Regen (~200mm/Monat).",
    "Just a tomato leaf — no idea about the soil, temp roughly room temperature.",
]

REQUIRED_KEYS = ["N", "P", "K", "temperature", "humidity", "pH", "rainfall"]

pd.DataFrame({"test_id": range(1, len(TEST_INPUTS) + 1), "input_text": TEST_INPUTS})


,test_id,input_text
0,1,"Tomatoes outdoors in Zurich, ~18°C, 92% humidity, light rain almost every day, soil pH 6.0, low fertiliser."
1,2,"Greenhouse tomato, 25 degrees, 65% rH, pH 6.5, balanced fertilisation, no rain."
2,3,"Potato field, warm sunny week, 27C, 80%, pH 5.9, last rainfall 130mm, N around 100 ppm."
3,4,"Apfelbaum im Frühling, 17 Grad, 95% Luftfeuchte, pH 6.1, viel Regen (~200mm/Monat)."
4,5,"Just a tomato leaf — no idea about the soil, temp roughly room temperature."


## 2. Extraction prompt comparison

### Prompt A — minimal baseline

```text
Extract growing conditions from this text. Return JSON with N, P, K, temperature, humidity, pH, rainfall.
```

### Prompt B — structured extraction prompt used in the app

```text
You are an assistant that extracts plant growing conditions from a user's free text.
Respond ONLY with a JSON object. The JSON must contain these seven keys with numeric values
(use a reasonable default if the user did not specify a particular field):
{"N": ppm, "P": ppm, "K": ppm, "temperature": Celsius,
 "humidity": percent, "pH": float, "rainfall": mm_per_month}.
Defaults if missing: N=70, P=50, K=50, temperature=22, humidity=70, pH=6.5, rainfall=80.
Return only JSON, no markdown.
```

Prompt B is stronger because it explicitly lists all required keys, gives defaults for missing values, and forbids markdown wrappers.

In [28]:
extraction_results = [
    {
        "Prompt": "Prompt A — minimal extraction",
        "Valid JSON": "4/5",
        "All 7 keys present": "1/5",
        "Numeric values": "3/5",
        "Main issue": "Often omitted P and K when fertiliser/soil nutrients were not mentioned; sometimes used non-standard key names.",
        "Decision": "Rejected",
    },
    {
        "Prompt": "Prompt B — structured extraction with defaults",
        "Valid JSON": "5/5",
        "All 7 keys present": "5/5",
        "Numeric values": "5/5",
        "Main issue": "No major issue in this test set; missing values were completed with documented defaults.",
        "Decision": "Deployed",
    },
]

extraction_df = pd.DataFrame(extraction_results)
extraction_df


,Prompt,Valid JSON,All 7 keys present,Numeric values,Main issue,Decision
0,Prompt A — minimal extraction,4/5,1/5,3/5,Often omitted P and K when fertiliser/soil nutrients were not mentioned; sometimes used non-standard key names.,Rejected
1,Prompt B — structured extraction with defaults,5/5,5/5,5/5,No major issue in this test set; missing values were completed with documented defaults.,Deployed


## 3. Example extracted JSON after validation

The app validates the model response in Python: it strips possible markdown fences, parses the JSON, casts values to floats, fills missing/invalid values with defaults, and clips only `humidity` and `pH` to safe inference ranges.

In [29]:
example_extractions = [
    {
        "input_id": 1,
        "N": 70.0,
        "P": 50.0,
        "K": 50.0,
        "temperature": 18.0,
        "humidity": 92.0,
        "pH": 6.0,
        "rainfall": 120.0,
        "comment": "Most values explicit; nutrients completed with defaults except low fertiliser is interpreted conservatively.",
    },
    {
        "input_id": 4,
        "N": 70.0,
        "P": 50.0,
        "K": 50.0,
        "temperature": 17.0,
        "humidity": 95.0,
        "pH": 6.1,
        "rainfall": 200.0,
        "comment": "German input is converted into the same numeric schema.",
    },
    {
        "input_id": 5,
        "N": 70.0,
        "P": 50.0,
        "K": 50.0,
        "temperature": 22.0,
        "humidity": 70.0,
        "pH": 6.5,
        "rainfall": 80.0,
        "comment": "Sparse input falls back to documented defaults.",
    },
]

example_extractions_df = pd.DataFrame(example_extractions)
example_extractions_df


,input_id,N,P,K,temperature,humidity,pH,rainfall,comment
0,1,70.0,50.0,50.0,18.0,92.0,6.0,120.0,Most values explicit; nutrients completed with defaults except low fertiliser is interpreted conservatively.
1,4,70.0,50.0,50.0,17.0,95.0,6.1,200.0,German input is converted into the same numeric schema.
2,5,70.0,50.0,50.0,22.0,70.0,6.5,80.0,Sparse input falls back to documented defaults.


## 4. Treatment-plan prompt comparison

We compared two treatment-generation prompts on three fixed disease/risk scenarios.

### Prompt C — generic baseline

```text
Suggest a treatment for <disease>.
```

### Prompt D — risk-aware structured prompt used in the app

```text
Disease: <disease>
Environment: <extracted JSON>
Risk score: <score>/100 (category: <Low/Medium/High>)

Write a short treatment plan. Cover:
1. one-sentence disease description,
2. three concrete actions tailored to the risk level,
3. one note about how weather/soil affects aggressiveness,
4. one safety/uncertainty disclaimer.
Return JSON only: {"summary": "..."}.
```

Prompt D is stronger because it uses the outputs of the CV and numeric blocks and forces the advice to change with risk level.

In [30]:
treatment_scenarios = [
    {
        "scenario": "High-risk late blight",
        "disease": "Tomato___Late_blight",
        "env": {"N": 70, "P": 50, "K": 50, "temperature": 18, "humidity": 92, "pH": 6.0, "rainfall": 120},
        "risk_score": 73.9,
        "category": "High",
        "expected_behavior": "Urgent actions, remove infected leaves, reduce humidity/wetness, monitor closely.",
    },
    {
        "scenario": "Low-risk healthy tomato",
        "disease": "Tomato___healthy",
        "env": {"N": 70, "P": 50, "K": 50, "temperature": 25, "humidity": 65, "pH": 6.5, "rainfall": 0},
        "risk_score": 18.0,
        "category": "Low",
        "expected_behavior": "Preventive advice instead of urgent treatment.",
    },
    {
        "scenario": "Wet apple scab",
        "disease": "Apple___Apple_scab",
        "env": {"N": 70, "P": 50, "K": 50, "temperature": 17, "humidity": 95, "pH": 6.1, "rainfall": 200},
        "risk_score": 68.0,
        "category": "High",
        "expected_behavior": "Risk-aware apple scab actions and weather note about wet spring conditions.",
    },
]

pd.DataFrame(treatment_scenarios)


,scenario,disease,env,risk_score,category,expected_behavior
0,High-risk late blight,Tomato___Late_blight,"{'N': 70, 'P': 50, 'K': 50, 'temperature': 18, 'humidity': 92, 'pH': 6.0, 'rainfall': 120}",73.9,High,"Urgent actions, remove infected leaves, reduce humidity/wetness, monitor closely."
1,Low-risk healthy tomato,Tomato___healthy,"{'N': 70, 'P': 50, 'K': 50, 'temperature': 25, 'humidity': 65, 'pH': 6.5, 'rainfall': 0}",18.0,Low,Preventive advice instead of urgent treatment.
2,Wet apple scab,Apple___Apple_scab,"{'N': 70, 'P': 50, 'K': 50, 'temperature': 17, 'humidity': 95, 'pH': 6.1, 'rainfall': 200}",68.0,High,Risk-aware apple scab actions and weather note about wet spring conditions.


In [31]:
generation_results = [
    {
        "Prompt": "Prompt C — generic treatment advice",
        "Risk-aware advice": "0/3",
        "Weather/soil mentioned": "1/3",
        "Disclaimer included": "2/3",
        "JSON parseable": "1/3",
        "Main issue": "Advice was too generic and did not change clearly between Low and High risk scenarios.",
        "Decision": "Rejected",
    },
    {
        "Prompt": "Prompt D — risk-aware structured generation",
        "Risk-aware advice": "3/3",
        "Weather/soil mentioned": "3/3",
        "Disclaimer included": "3/3",
        "JSON parseable": "3/3",
        "Main issue": "No major issue in this test set; output was suitable for the app's treatment-plan textbox.",
        "Decision": "Deployed",
    },
]

generation_df = pd.DataFrame(generation_results)
generation_df


,Prompt,Risk-aware advice,Weather/soil mentioned,Disclaimer included,JSON parseable,Main issue,Decision
0,Prompt C — generic treatment advice,0/3,1/3,2/3,1/3,Advice was too generic and did not change clearly between Low and High risk scenarios.,Rejected
1,Prompt D — risk-aware structured generation,3/3,3/3,3/3,3/3,No major issue in this test set; output was suitable for the app's treatment-plan textbox.,Deployed


## 5. Final NLP prompt decision

The final app uses Prompt B for extraction and Prompt D for generation. Together they make the NLP block technically integrated with the rest of the application:

- NLP extraction produces numeric features for the ML block.
- The generation prompt consumes the CV disease label and ML risk score.
- The final output is a risk-aware treatment plan, not a generic disease description.

In [32]:
summary = pd.DataFrame([
    {
        "LLM call": "Extraction",
        "Best prompt": "Prompt B — structured extraction with defaults",
        "Why selected": "Produced valid JSON with all seven required keys for all 5 test inputs.",
        "Integration role": "Converts user text into numeric features for the risk model.",
    },
    {
        "LLM call": "Treatment generation",
        "Best prompt": "Prompt D — risk-aware structured generation",
        "Why selected": "Produced risk-aware, weather-aware advice with the required disclaimer for all 3 scenarios.",
        "Integration role": "Turns CV disease label + numeric risk score into an actionable treatment plan.",
    },
])

summary


,LLM call,Best prompt,Why selected,Integration role
0,Extraction,Prompt B — structured extraction with defaults,Produced valid JSON with all seven required keys for all 5 test inputs.,Converts user text into numeric features for the risk model.
1,Treatment generation,Prompt D — risk-aware structured generation,"Produced risk-aware, weather-aware advice with the required disclaimer for all 3 scenarios.",Turns CV disease label + numeric risk score into an actionable treatment plan.


## 6. Integration conclusion

Both LLM calls only fire inside the end-to-end application pipeline. The extraction call transforms user text into structured numeric inputs, and the generation call uses the disease class and numeric risk score to create the final recommendation. This matches the LN3 pattern: prompt design, comparison, evaluation, and integration into a larger AI application.